<a href="https://colab.research.google.com/github/gurbaaz2599bbafai25-commits/subscription-conversion-prediction-ml/blob/main/Subscription_Conversion_Prediction_ML_Project_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Subscription Conversion Prediction — End-to-End ML Project

## Business Problem
A subscription-based app has many free users, but only a portion of them convert to paid subscriptions. The business wants to **predict which free users are likely to become paid customers** using their engagement behavior.

This can help the business:
- Identify high-potential free users
- Prioritize conversion campaigns
- Personalize offers and messaging
- Improve marketing efficiency
- Understand which engagement signals are associated with conversion

## Machine Learning Problem
This is a **supervised binary classification** problem.

### Target Variable
`Converted`

- `1` → User became a paid subscriber
- `0` → User did not become a paid subscriber

### Input Features
- Age
- Days since signup
- Sessions
- Visits
- Features used
- Average session duration
- Trial days used
- Support interactions

## Models Used
We will train and compare three classification algorithms:

1. **Logistic Regression** — interpretable linear baseline
2. **Decision Tree Classifier** — captures non-linear decision rules
3. **Random Forest Classifier** — ensemble of multiple decision trees

## Evaluation Metrics
Models will be compared using:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

## Final Architecture

```text
Raw User Data
      ↓
Data Cleaning
      ↓
EDA
      ↓
Feature Engineering
      ↓
Train / Test Split
      ↓
 ┌────────────────────────────┐
 │ Logistic Regression        │
 │ Decision Tree              │
 │ Random Forest              │
 └────────────────────────────┘
      ↓
Model Evaluation
      ↓
Accuracy | Precision | Recall
F1 | ROC-AUC | Confusion Matrix
      ↓
Select / Tune Model
      ↓
Predict Conversion Probability
```
Pipeline
Data Collection → Data Understanding → Data Cleaning → Outlier Detection & Treatment → EDA → Feature Engineering → Define Target Variable → Feature Selection → Encode Target Variable → Train-Test Split → Feature Standardisation → Model Building → Model Training → Prediction → Model Evaluation → Best Model Selection → Model Interpretation → Prediction & Reporting → Final Output

## 1. Data Source & Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report


## 2. Data Ingestion
Upload the subscription conversion CSV file in Google Colab.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Data Understanding

In [ ]:
# Explore the raw data before making any changes to it
print("Shape of raw data:", df.shape)

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nSummary statistics:")
display(df.describe())

print("\nDuplicate rows found:", df.duplicated().sum())

print("\nMissing values per column:")
display(df.isnull().sum().to_frame("Missing Count"))

print("\nTarget class distribution:")
display(df["Converted"].value_counts().rename({0: "Not Converted", 1: "Converted"}))


## 4. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Fill missing numerical values with the median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)
print("Total missing values after cleaning:", df.isnull().sum().sum())


## 5. Outlier Detection & Treatment

In [ ]:
# Detect outliers using the IQR method and treat them by capping (winsorizing)
# rather than dropping rows, since the dataset is small and every user record matters
outlier_cols = [
    "Age", "Days_Since_Signup", "Sessions", "Visits",
    "Features_Used", "Avg_Session_Minutes", "Trial_Days_Used",
    "Support_Interactions"
]

outlier_summary = []

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_summary.append({
        "Feature": col,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outliers Found": n_outliers
    })

    # Cap values outside the IQR bounds
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)
print("\nOutliers capped to the lower/upper IQR bounds shown above.")


## 6. EDA — Exploratory Data Analysis

In [ ]:
# Basic dataset overview
print("Data types:")
display(df.dtypes.to_frame("Data Type"))

print("\nTarget distribution:")
display(df["Converted"].value_counts().rename({0: "Not Converted", 1: "Converted"}))

print("\nSummary statistics:")
display(df.describe())


In [ ]:
# Target distribution
df["Converted"].value_counts().sort_index().plot(kind="bar")
plt.title("Subscription Conversion Distribution")
plt.xlabel("Converted")
plt.ylabel("Number of Users")
plt.xticks([0, 1], ["Not Converted", "Converted"], rotation=0)
plt.show()


In [ ]:
# Relationship between engagement variables and conversion
eda_cols = [
    "Sessions", "Visits", "Features_Used",
    "Avg_Session_Minutes", "Trial_Days_Used",
    "Support_Interactions"
]

for col in eda_cols:
    df.groupby("Converted")[col].mean().plot(kind="bar")
    plt.title(f"Average {col} by Conversion")
    plt.xlabel("Converted")
    plt.ylabel(col)
    plt.xticks([0, 1], ["Not Converted", "Converted"], rotation=0)
    plt.show()


## 7. Feature Engineering

In [ ]:
# Create simple engagement-rate features
df["Sessions_Per_Day"] = df["Sessions"] / df["Days_Since_Signup"].clip(lower=1)
df["Visits_Per_Day"] = df["Visits"] / df["Days_Since_Signup"].clip(lower=1)

display(df.head())


## 8. Define Target Variable

In [ ]:
target = "Converted"
y = df[target]

print("Target variable:", target)
print("\nClass distribution:")
display(y.value_counts().rename({0: "Not Converted", 1: "Converted"}))


## 9. Feature Selection

In [ ]:
# User_ID is an identifier, not a predictive feature.
# The target ("Converted") was already defined in the previous step.
feature_cols = [
    "Age",
    "Days_Since_Signup",
    "Sessions",
    "Visits",
    "Features_Used",
    "Avg_Session_Minutes",
    "Trial_Days_Used",
    "Support_Interactions",
    "Sessions_Per_Day",
    "Visits_Per_Day"
]

X = df[feature_cols]

print("Selected features:")
print(feature_cols)


## 10. Encode Target Variable

In [ ]:
# Converted is already numeric (0/1), but we encode it explicitly with
# LabelEncoder so the pipeline works unchanged even if the raw target were
# provided as text labels (e.g. "Yes"/"No")
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Target classes:", list(le.classes_))
print("Encoded mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


## 11. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 12. Feature Standardisation

In [ ]:
# Fit the scaler on the training data only, then apply it to both sets
# to avoid leaking test-set information into training
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_scaled.head())


## 13. Model Building

In [ ]:
# Define the candidate model objects (not yet trained).
# Features are already standardised, so Logistic Regression no longer
# needs its own internal scaling step.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
}

print("Models defined:", list(models.keys()))


## 14. Model Training

In [ ]:
for name, model in models.items():
    model.fit(X_train_scaled, y_train)

print("All models trained successfully.")


## 15. Prediction

In [ ]:
# Generate predictions and probabilities for every trained model on the test set
predictions = {}

for name, model in models.items():
    pred = model.predict(X_test_scaled)
    prob = model.predict_proba(X_test_scaled)[:, 1]
    predictions[name] = {"pred": pred, "prob": prob}

print("Predictions generated for:", list(predictions.keys()))


## 16. Model Evaluation

16.a. Confusion Matrix

In [ ]:
# Confusion Matrix comparison across all trained models
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))

labels = ["Not Converted", "Converted"]

for ax, (name, model) in zip(axes, models.items()):
    pred = predictions[name]["pred"]
    cm = confusion_matrix(y_test, pred)

    im = ax.imshow(cm, cmap="Blues")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(name)

    # Annotate each cell with its count
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.tight_layout()
plt.show()

16.b. Comparison

In [ ]:
results = []

for name, model in models.items():
    pred = predictions[name]["pred"]
    prob = predictions[name]["prob"]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(results_df)


## 17. Best Model Selection

In [ ]:
# Select the best model based on F1 Score (change the metric below if a different
# business priority — e.g. Recall to catch more potential converters — is preferred)
selection_metric = "F1 Score"

best_model_name = results_df.sort_values(selection_metric, ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
best_pred = predictions[best_model_name]["pred"]
best_prob = predictions[best_model_name]["prob"]

print(f"Best model selected (by {selection_metric}): {best_model_name}")
print("\nMetrics for selected model:")
display(results_df[results_df["Model"] == best_model_name])


## 18. Model Interpretation
Understanding *why* the best model makes the predictions it does — which features it relies on, and where its errors occur.

### 18a. Feature Importance (Best Model)

In [ ]:
# Works regardless of which model was selected:
# - Tree-based models (Decision Tree, Random Forest) expose feature_importances_
# - Logistic Regression exposes coefficients instead, so we use their
#   absolute value as an importance proxy

if hasattr(best_model, "feature_importances_"):
    importance_values = best_model.feature_importances_
    importance_label = "Importance"
elif hasattr(best_model, "coef_"):
    importance_values = np.abs(best_model.coef_[0])
    importance_label = "Importance (|coefficient|)"
else:
    raise ValueError("Selected model does not expose feature importances or coefficients.")

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    importance_label: importance_values
}).sort_values(importance_label, ascending=False).reset_index(drop=True)

display(importance_df)


In [ ]:
importance_df.plot(
    x="Feature",
    y=importance_label,
    kind="bar",
    legend=False,
    figsize=(9, 5)
)
plt.title(f"{best_model_name} Feature Importance")
plt.xlabel("Feature")
plt.ylabel(importance_label)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 18b. Confusion Matrix (Best Model)

In [ ]:
print("Selected model:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, best_pred, zero_division=0))

cm = confusion_matrix(y_test, best_pred)
print("Confusion Matrix (raw counts):")
print(cm)


In [ ]:
# Visual confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")

labels = ["Not Converted", "Converted"]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_model_name}")

# Annotate each cell with its count
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im, ax=ax, label="Count")
plt.tight_layout()
plt.show()


## 19. Prediction & Reporting

In [ ]:
# Predict conversion probability for the test users
# (original, unscaled feature values are used here so the report stays human-readable)
prediction_output = X_test.copy()
prediction_output["Actual_Converted"] = y_test
prediction_output["Predicted_Converted"] = best_pred
prediction_output["Conversion_Probability"] = best_prob

display(prediction_output.head(10))


In [ ]:
# Simple business segmentation
prediction_output["Segment"] = pd.cut(
    prediction_output["Conversion_Probability"],
    bins=[-0.01, 0.33, 0.66, 1.0],
    labels=["Low Potential", "Medium Potential", "High Potential"]
)

display(
    prediction_output[
        ["Actual_Converted", "Predicted_Converted", "Conversion_Probability", "Segment"]
    ].head(10)
)


## 20. Final Output

### Business Objective
Predict which free users are likely to convert to paid subscribers.

### Final Pipeline
**Data Collection → Data Understanding → Data Cleaning → Outlier Detection & Treatment → EDA → Feature Engineering → Define Target Variable → Feature Selection → Encode Target Variable → Train-Test Split → Feature Standardisation → Model Building → Model Training → Prediction → Model Evaluation → Best Model Selection → Model Interpretation → Prediction & Reporting → Final Output**

### Output
The project produces:
- Raw-data understanding (shape, types, missing values, duplicates, class balance)
- Cleaned data (duplicates removed, missing values imputed)
- Outlier-treated features (capped using the IQR method)
- EDA insights
- Engineered features
- An explicitly defined and encoded target variable
- Selected input features
- Standardised features (fit on training data only)
- Model objects, trained
- Test-set predictions for every candidate model
- Model evaluation metrics for all candidate models
- The selected best-performing model
- Feature importance and a confusion matrix (report + heatmap) for the selected model
- Final conversion predictions, probabilities, and business segmentation
